## Importing libraries


In [2]:
# Import Polars for dataframe operations
import polars as pl

# Import Path for managing file paths
from pathlib import Path

## Define Data folder


In [3]:
# Define path to data folder
data_folder = Path("../data")

## Load Cleaned Dataset (Parquet Version)


In [ ]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Check data types
print(sales_df.schema)

# preview data set
sales_df.head()

print("Rows:", sales_df.height)
print("Columns:", sales_df.width)

Schema({'transaction_id': String, 'customer_id': String, 'product_id': String, 'purchase_date': Datetime(time_unit='us', time_zone=None), 'festival': String, 'city': String, 'product_name': String, 'category': String, 'customer_name': String, 'age': Int64, 'gender': String, 'price': Int64, 'revenue': Int64})
Rows: 50000
Columns: 13


## Find Latest Purchase Date


In [10]:
# Find the most recent purchase date
latest_date = sales_df.select(pl.col("purchase_date").max()).item()
latest_date

datetime.datetime(2024, 12, 31, 0, 0)

## Calculate Recency

Recency = Days since last purchase


In [12]:
# Get last purchase date per customer
receny_df = sales_df.group_by("customer_id").agg(
    pl.col("purchase_date").max().alias("last_purchase")
)

# Calculate recency

receny_df = receny_df.with_columns(
    (latest_date - pl.col("last_purchase")).dt.total_days().alias("recency")
)

receny_df.head()

customer_id,last_purchase,recency
str,datetime[μs],i64
"""c667""",2024-12-31 00:00:00,0
"""c521""",2024-12-26 00:00:00,5
"""c204""",2024-12-13 00:00:00,18
"""c96""",2024-12-17 00:00:00,14
"""c420""",2024-12-24 00:00:00,7


## Calculate Frequency

Frequency = number of purchases


In [13]:
# Count purchases per customer

frequency_df = sales_df.group_by("customer_id").agg(
    pl.count().alias("frequency")
)

frequency_df.head()

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_3456\2495761181.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("frequency")


customer_id,frequency
str,u32
"""c538""",79
"""c349""",45
"""c459""",58
"""c688""",61
"""c775""",55


## Calculate Monetary Value

Monetary = total revenue spent


In [17]:
# Calculate total revenue per customer

monetary_df = sales_df.group_by("customer_id").agg(
    pl.col("revenue").sum().alias("monetary")
)

monetary_df.head()

customer_id,monetary
str,i64
"""c517""",16845
"""c840""",24627
"""c994""",18437
"""c607""",20886
"""c105""",20239


## Merge RFM Metrics


In [18]:
# Merge recency, frequency and monetary tables
rfm_df = receny_df.join(frequency_df, on="customer_id").join(
    monetary_df, on="customer_id"
)

rfm_df.head()

customer_id,last_purchase,recency,frequency,monetary
str,datetime[μs],i64,u32,i64
"""c517""",2024-12-28 00:00:00,3,55,16845
"""c840""",2024-12-29 00:00:00,2,73,24627
"""c994""",2024-12-30 00:00:00,1,63,18437
"""c607""",2024-12-27 00:00:00,4,64,20886
"""c105""",2024-12-15 00:00:00,16,61,20239


## Create RFM Scores


In [20]:
# Create RFM scores using rank

rfm_df = rfm_df.with_columns(
    [
        pl.col("recency").rank("dense").alias("r_score"),
        pl.col("frequency").rank("dense").alias("f_score"),
        pl.col("monetary").rank("dense").alias("m_score"),
    ]
)

rfm_df.head()

customer_id,last_purchase,recency,frequency,monetary,r_score,f_score,m_score
str,datetime[μs],i64,u32,i64,u32,u32,u32
"""c517""",2024-12-28 00:00:00,3,55,16845,4,47,492
"""c840""",2024-12-29 00:00:00,2,73,24627,3,65,806
"""c994""",2024-12-30 00:00:00,1,63,18437,2,55,581
"""c607""",2024-12-27 00:00:00,4,64,20886,5,56,694
"""c105""",2024-12-15 00:00:00,16,61,20239,17,53,675


## Create RFM Score


In [ ]:
# Combine RFM scores

rfm_df = rfm_df.with_columns(
    (
        pl.col("r_score") +
        pl.col("f_score") +
        pl.col("m_score")
    )
    .alias("rfm_score")
)

rfm_df.head()

customer_id,last_purchase,recency,frequency,monetary,r_score,f_score,m_score,rfm_score
str,datetime[μs],i64,u32,i64,u32,u32,u32,u32
"""c517""",2024-12-28 00:00:00,3,55,16845,543,47,492,1082
"""c840""",2024-12-29 00:00:00,2,73,24627,874,65,806,1745
"""c994""",2024-12-30 00:00:00,1,63,18437,638,55,581,1274
"""c607""",2024-12-27 00:00:00,4,64,20886,755,56,694,1505
"""c105""",2024-12-15 00:00:00,16,61,20239,745,53,675,1473


## Create Customer Segments


In [ ]:
# Assign customer segments

rfm_df = rfm_df.with_columns(
    pl.when(pl.col("rfm_score") >= 1500).then(pl.lit("Champions"))
    .when(pl.col("rfm_score") >= 1200).then(pl.lit("Loyal Customers"))
    .when(pl.col("rfm_score") >= 600).then(pl.lit("Potential Loyalists"))
    .otherwise(pl.lit("At Risk"))
    .alias("segment")
)

rfm_df.head()

customer_id,last_purchase,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment
str,datetime[μs],i64,u32,i64,u32,u32,u32,u32,str
"""c517""",2024-12-28 00:00:00,3,55,16845,543,47,492,1082,"""Potential Loyalists"""
"""c840""",2024-12-29 00:00:00,2,73,24627,874,65,806,1745,"""Champions"""
"""c994""",2024-12-30 00:00:00,1,63,18437,638,55,581,1274,"""Loyal Customers"""
"""c607""",2024-12-27 00:00:00,4,64,20886,755,56,694,1505,"""Champions"""
"""c105""",2024-12-15 00:00:00,16,61,20239,745,53,675,1473,"""Loyal Customers"""


## Segmet Distribution


In [34]:
# Count customers per segment

rfm_df.group_by("segment").count().sort("count", descending=True)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_3456\3515162173.py:3: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  rfm_df.group_by("segment").count().sort("count", descending=True)


segment,count
str,u32
"""Potential Loyalists""",327
"""At Risk""",302
"""Champions""",206
"""Loyal Customers""",165


## Save Dataset


In [ ]:
# Save RFM dataframe as csv
rfm_df.write_csv(data_folder / "customer_rfm.csv")

In [36]:
# Save RFM dataframe as parquet
rfm_df.write_parquet(data_folder / "customer_rfm.parquet")

# Sanity Check

In [37]:
rfm_df.describe()

statistic,customer_id,last_purchase,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment
str,str,str,f64,f64,f64,f64,f64,f64,f64,str
"""count""","""1000""","""1000""",1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,"""1000"""
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0"""
"""mean""",null,"""2024-12-21 17:00:57.600000""",9.291,50.0,16245.95,490.591,41.96,438.415,970.966,null
"""std""",null,null,10.69461,17.516702,5803.458224,267.482902,17.382363,253.909698,538.144211,null
"""min""","""c0""","""2024-10-23 00:00:00""",0.0,7.0,2493.0,6.0,1.0,1.0,8.0,"""At Risk"""
"""25%""",null,"""2024-12-18 00:00:00""",2.0,38.0,12214.0,263.0,30.0,219.0,513.0,null
"""50%""",null,"""2024-12-26 00:00:00""",6.0,49.0,15955.0,490.0,41.0,439.0,973.0,null
"""75%""",null,"""2024-12-29 00:00:00""",13.0,61.0,19691.0,712.0,53.0,653.0,1416.0,null
"""max""","""c999""","""2024-12-31 00:00:00""",69.0,110.0,36590.0,988.0,93.0,891.0,1972.0,"""Potential Loyalists"""


In [38]:
rfm_df.group_by("segment").count()

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_3456\84912235.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  rfm_df.group_by("segment").count()


segment,count
str,u32
"""Loyal Customers""",165
"""Champions""",206
"""At Risk""",302
"""Potential Loyalists""",327
